In [5]:
import torch
import numpy as np
from datasets import load_dataset
from tqdm.auto import tqdm
from torchvision.models import resnet50, ResNet50_Weights
import warnings

In [7]:
def bit2float(b, num_e_bits=8, num_m_bits=23, bias=127.):
    """Turn input tensor into float.

        Args:
            b : binary tensor. The last dimension of this tensor should be the
            the one the binary is at.
            num_e_bits : Number of exponent bits. Default: 8.
            num_m_bits : Number of mantissa bits. Default: 23.
            bias : Exponent bias/ zero offset. Default: 127.
        Returns:
            Tensor: Float tensor. Reduces last dimension.

    """
    expected_last_dim = num_m_bits + num_e_bits + 1
    assert b.shape[-1] == expected_last_dim, "Binary tensors last dimension " \
                                             "should be {}, not {}.".format(
        expected_last_dim, b.shape[-1])

    # check if we got the right type
    dtype = torch.float32
    if expected_last_dim > 32: dtype = torch.float64
    if expected_last_dim > 64:
        warnings.warn("pytorch can not process floats larger than 64 bits, keep"
                      " this in mind. Your result will be not exact.")

    s = torch.index_select(b, -1, torch.arange(0, 1))
    e = torch.index_select(b, -1, torch.arange(1, 1 + num_e_bits))
    m = torch.index_select(b, -1, torch.arange(1 + num_e_bits,
                                               1 + num_e_bits + num_m_bits))
    # SIGN BIT
    out = ((-1) ** s).squeeze(-1).type(dtype)
    # EXPONENT BIT
    exponents = -torch.arange(-(num_e_bits - 1.), 1.)
    exponents = exponents.repeat(b.shape[:-1] + (1,))
    e_decimal = torch.sum(e * 2 ** exponents, dim=-1) - bias
    out *= 2 ** e_decimal
    # MANTISSA
    matissa = (torch.Tensor([2.]) ** (
        -torch.arange(1., num_m_bits + 1.))).repeat(
        m.shape[:-1] + (1,))
    out *= 1. + torch.sum(m * matissa, dim=-1)
    return out


def float2bit(f, num_e_bits=8, num_m_bits=23, bias=127., dtype=torch.float32):
    """Turn input tensor into binary.

        Args:
            f : float tensor.
            num_e_bits : Number of exponent bits. Default: 8.
            num_m_bits : Number of mantissa bits. Default: 23.
            bias : Exponent bias/ zero offset. Default: 127.
            dtype : This is the actual type of the tensor that is going to be
            returned. Default: torch.float32.
        Returns:
            Tensor: Binary tensor. Adds last dimension to original tensor for
            bits.

    """
    ## SIGN BIT
    s = torch.sign(f)
    f = f * s
    # turn sign into sign-bit
    s = (s * (-1) + 1.) * 0.5
    s[s == 0.5] = 1
    s = s.unsqueeze(-1)

    ## EXPONENT BIT
    e_scientific = torch.floor(torch.log2(f))
    e_decimal = e_scientific + bias
    e = integer2bit(e_decimal, num_bits=num_e_bits)
    e[torch.isnan(e)] = 0

    ## MANTISSA
    m1 = integer2bit(f - f % 1, num_bits=num_e_bits)
    m2 = remainder2bit(f % 1, num_bits=bias)
    m = torch.cat([m1, m2], dim=-1)

    dtype = f.type()
    idx = torch.arange(num_m_bits).unsqueeze(0).type(dtype) \
          + (8. - e_scientific).unsqueeze(-1)
    idx = idx.long()
    idx[idx == -9223372036854775808] = 0
    m = torch.gather(m, dim=-1, index=idx)

    return torch.cat([s, e, m], dim=-1).type(dtype)


def remainder2bit(remainder, num_bits=127):
    """Turn a tensor with remainders (floats < 1) to mantissa bits.

        Args:
            remainder : torch.Tensor, tensor with remainders
            num_bits : Number of bits to specify the precision. Default: 127.
        Returns:
            Tensor: Binary tensor. Adds last dimension to original tensor for
            bits.

    """
    dtype = remainder.type()
    exponent_bits = torch.arange(num_bits).type(dtype)
    exponent_bits = exponent_bits.repeat(remainder.shape + (1,))
    out = (remainder.unsqueeze(-1) * 2 ** exponent_bits) % 1
    return torch.floor(2 * out)


def integer2bit(integer, num_bits=8):
    """Turn integer tensor to binary representation.

        Args:
            integer : torch.Tensor, tensor with integers
            num_bits : Number of bits to specify the precision. Default: 8.
        Returns:
            Tensor: Binary tensor. Adds last dimension to original tensor for
            bits.

    """
    dtype = integer.type()
    exponent_bits = -torch.arange(-(num_bits - 1), 1).type(dtype)
    exponent_bits = exponent_bits.repeat(integer.shape + (1,))
    out = integer.unsqueeze(-1) / 2 ** exponent_bits
    return (out - (out % 1)) % 2

In [8]:
def add_noise_float_layer(prob,mask,layer):
    pred = float2bit(layer, num_e_bits=8, num_m_bits=23, bias=127.)
    shape = pred.shape
    pred = pred.reshape(-1)
    mask_length = pred.numel()
    repeats = mask_length // len(mask)
    duplicated_array = np.tile(mask, repeats)[:mask_length]
    mask = torch.IntTensor(duplicated_array)

    noise_mask = torch.rand(mask_length)
    noise_mask = (noise_mask < prob)

    noise_weight_binary = pred.int() ^ (noise_mask.int() & (~mask))
    noise_weight_binary = noise_weight_binary.reshape(shape)
    noise_weight = bit2float(noise_weight_binary)
    return noise_weight

In [9]:
conv_layer_names = [
    'conv1.weight',
    'layer1.0.conv1.weight',
    'layer1.0.conv2.weight',
    'layer1.1.conv1.weight',
    'layer1.1.conv2.weight',
    'layer2.0.conv1.weight',
    'layer2.0.conv2.weight',
    'layer2.0.downsample.0.weight',
    'layer2.1.conv1.weight',
    'layer2.1.conv2.weight',
    'layer3.0.conv1.weight',
    'layer3.0.conv2.weight',
    'layer3.0.downsample.0.weight',
    'layer3.1.conv1.weight',
    'layer3.1.conv2.weight',
    'layer4.0.conv1.weight',
    'layer4.0.conv2.weight',
    'layer4.0.downsample.0.weight',
    'layer4.1.conv1.weight',
    'layer4.1.conv2.weight',
    'fc.weight',
]

In [11]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from copy import deepcopy
import time

prob_list = [1e-10, 5e-10, 1e-9,5e-9, 1e-8, 5e-8, 1e-7, 5e-7, 1e-6, 5e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1, 5e-1]


# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define data transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Download CIFAR-10 dataset for evaluation
eval_dataset = datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False, num_workers=4)

# Create ResNet18 model
model = models.resnet18(pretrained=False)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)  # Adjust the last layer for 10 classes in CIFAR-10

# Load the pre-trained weights
model_path = 'resnet18_cifar10_epoch_3000.pth'  # Replace with the path to your saved model
model.load_state_dict(torch.load(model_path, map_location=device))


mask = [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]

orignal_dict = model.state_dict()

model = model.to(device)

total_acc = []

for i in range(0,len(conv_layer_names)):

    final_acc_list =[]
    noise_layer_for_testing = []
    noise_layer_for_testing.append(conv_layer_names[i])

    for j in range(0,10):
        acc_list = []
        for prob in prob_list:

            params = deepcopy(orignal_dict)

            for layer_name in noise_layer_for_testing:

                layer_weights = params[layer_name]
                noise_weight = add_noise_float_layer(prob,mask,layer_weights)
                params[layer_name] = noise_weight

            model.load_state_dict(params)

            # Evaluate the model on the test set
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for inputs, labels in eval_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

            accuracy = correct / total
            acc_list.append(accuracy)
        total_acc.append(acc_list)
        print(acc_list)
    final_acc_list.append(total_acc)


file_path = "acc_list.pkl"

# Write the list to disk
with open(file_path, 'wb') as file:
    pickle.dump(final_acc_list, file)

Files already downloaded and verified
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
[0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7863, 0.0984, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
start inference
[0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7873, 0.7874, 0.7864, 0.7867, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
start inference
start inference
start infe

NameError: name 'pickle' is not defined

In [ ]:
import pickle

#print(total_acc)
file_path = "acc_list.pkl"

# Write the list to disk
with open(file_path, 'wb') as file:
    pickle.dump(final_acc_list, file)
print(len(total_acc))

reshaped_list = [total_acc[i:i+10] for i in range(0, 210, 10)]

# Now, create a list containing 21 lists, each containing 10 lists
final_list = [reshaped_list[i:i+10] for i in range(0, 21, 1)]

In [36]:
def calculate_average(lst):
    return sum(lst) / len(lst)

# Empty list to store averages
averages = []

# Iterating over every 10 lists
for i in range(0, len(total_acc), 10):
    # Extracting 10 lists
    sublist = total_acc[i:i+10]
    
    # Transposing the sublists to calculate column-wise averages
    transposed_sublist = list(zip(*sublist))
    
    # Calculating average of each sublist
    sublist_averages = [calculate_average(column) for column in transposed_sublist]
    
    # Appending sublist averages to the result list
    averages.append(sublist_averages)

with open("averages.txt", "w") as file:
    for sublist in averages:
        file.write(' '.join(map(str, sublist)) + '\n')

TypeError: unsupported operand type(s) for +=: 'int' and 'list'